## 🇺🇦 Wartime Civilian Harms and Adolescent Trauma in Ukraine

_WIP - NOT FOR DISTRIBUTION_

**Preregistration and STROBE checklist in progress**

*Provided in the spirit of open science. This notebook offers a "one-click" replication wrangling and building the _oblast_ (region) and _raion_ (district) Bellingcat OSINT [Civilian Harm in Ukraine](https://ukraine.bellingcat.com/) geocoded event-level data $\rightarrow$ Ukraine Longitudinal Study (ULS) and (non-replicable) individual-level _Ukraine Longitudinal Survey_ (ULS) 1:$n$ merge. To protect the safety and anonymity of the child-adolescent ULS respondents, those data are private and cannot be released. 

⛏️ `uls_scratchpad.ipynb`<br>
Simone J. Skeen x Claude Code (07-29-2026)

1. [Prepare](#1-prepare)
2. [Import + transform: Bellingcat OSINT Civilian Harm in Ukraine](#2-import--transform-bellingcat-osint-civilian-harm-in-ukraine)
3. [Import + transform: Uppsala Conflict Data Program](#3-import--transform-uppsala-conflict-data-program)


### 1. Prepare
Imports requisite packages; customizes outputs.
***
**Dependencies:** Install via `pip install -r requirements.txt` from project root before running.

In [1]:
%%capture

%pip install -r ../../requirements.txt

In [2]:
# Standard library
import json
import os
import re
import sys
import urllib.request
import warnings
import zipfile
from datetime import datetime
from pathlib import Path
from time import sleep

# Add src to path for local imports
sys.path.insert(0, str(Path.cwd().parent))

# Third-party
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv
from geopy.geocoders import Nominatim
from IPython.core.interactiveshell import InteractiveShell
from tqdm.notebook import tqdm

# Local
from mappings import ADMIN_UNIT_TO_OBLAST, RAION_UA_TO_EN

In [3]:
# Env variables
load_dotenv()
BELLINGCAT_API_URL = os.getenv('BELLINGCAT_API_URL')
UCDP_GED_URL = os.getenv('UCDP_GED_URL')

# Output preferences
InteractiveShell.ast_node_interactivity = 'all'

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

In [4]:
%%script false --no-raise-error

# Project directory structure
.
└── civilian-trauma/
    ├── config
    ├── data/
    │   ├── raw/
    │   │   ├── level_1
    │   │   └── level_2
    │   └── processed
    ├── src/
    │   └── notebooks
    └── outputs/
        └── figures

In [5]:
# Set working directory to project root; define data paths
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')

# Inputs subdirectories
DATA_RAW = 'data/raw'
DATA_PROC = 'data/processed'
DATA_LVL1 = f'{DATA_RAW}/level_1'   ### ULS child-adolescent survey data; not for public use
DATA_LVL2 = f'{DATA_RAW}/level_2'   ### Bellingcat geotagged OSINT data

# Ensure directories exist
for path in [DATA_RAW, DATA_PROC, DATA_LVL1, DATA_LVL2]:
    os.makedirs(path, exist_ok=True)

In [6]:
# Bellingcat data source
BELLINGCAT_CSV = 'ukr-civharm-2026-01-09.csv'

# ULS merge params

### TODO 7/29: update start date constant w/ new Wave 2 codebook

SURVEY_START_DATE = '2025-04-08' ### earliest observation in ULS Wave 2 survey
DATE_FORMAT_INPUT = '%m/%d/%Y'   ### format in source data (if .CSV fallback)
DATE_FORMAT_ISO = '%Y-%m-%d'     ### ISO 8601 for internal use

# Ukrposhta postcode directory (data.gov.ua)
POSTCODE_7Z = 'zvit-dlia-miu-perelik-poshtovikh-indeksiv-ta-viddilen_08-08-2025-csv.7z'

# Geocoding
NOMINATIM_USER_AGENT = 'ukraine_postcode_geocoder'
NOMINATIM_DELAY_SEC = 1  ### 1-second delay; ensures rate limit compliance

### 2. Import + transform: Bellingcat OSINT Civilian Harm in Ukraine
Imports, cleans, describes level-2 aggregate conflict data. Acquired via API (cf `.env`)

In [7]:
# Fetch routinely updated .JSON from Bellingcat API endpoint

### NOTE 7/29: `d_api.csv` & `d_dl.csv` are for visual inspection/human verification and can be deleted for prod

### docs: https://github.com/bellingcat/ukraine-timemap

def fetch_bellingcat_json(url):
    """
    Fetches Bellingcat civilian harm data from API endpoint.
    Returns list of event dictionaries.
    """
    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read().decode('utf-8'))
    return data

def convert_to_csv_format(events):
    """
    Converts Bellingcat API JSON to CSV format matching ukr-civharm-*.csv structure.
    
    JSON format: id, date (YYYY-MM-DD), latitude, longitude, location, 
                 description, sources (array), impact (array), weapon_system (array)
    CSV format:  id, date (MM/DD/YYYY), latitude, longitude, location,
                 description, sources (comma-sep), associations (formatted string)
    """
    rows = []
    for event in events:
        # Convert date: YYYY-MM-DD → MM/DD/YYYY
        date_iso = event.get('date', '')
        try:
            date_obj = datetime.strptime(date_iso, '%Y-%m-%d')
            date_formatted = date_obj.strftime('%m/%d/%Y')
        except ValueError:
            date_formatted = date_iso
        
        # Join sources array
        sources = event.get('sources', [])
        sources_str = ','.join(sources) if sources else ''
        
        # Build `associations` string from `impact` & `weapon_system`
        associations_parts = []
        for impact in event.get('impact', []):
            associations_parts.append(f'Type of area affected={impact}')
        for weapon in event.get('weapon_system', []):
            associations_parts.append(f'Weapon System={weapon}')
        associations_str = ','.join(associations_parts) if associations_parts else ''
        
        rows.append({
            'id': event.get('id', ''),
            'date': date_formatted,
            'latitude': event.get('latitude', ''),
            'longitude': event.get('longitude', ''),
            'location': (event.get('location') or '').strip(),
            'description': (event.get('description') or '').strip(),
            'sources': sources_str,
            'associations': associations_str,
        })
    
    return pd.DataFrame(rows)

# Fetch
print(f"Fetching data from Bellingcat API...")
d_lvl2_bcat_raw_json = fetch_bellingcat_json(BELLINGCAT_API_URL)

# Save raw .JSON 
json_path = f"{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(d_lvl2_bcat_raw_json, f, ensure_ascii=False, indent=2)
print(f"Raw JSON saved to: {json_path}")

# Convert & save to .CSV
d_lvl2_bcat_raw_csv = convert_to_csv_format(d_lvl2_bcat_raw_json)

# Sort by date (earliest first)
d_lvl2_bcat_raw_csv['_date_sort'] = pd.to_datetime(d_lvl2_bcat_raw_csv['date'], format='%m/%d/%Y', errors='coerce')
d_lvl2_bcat_raw_csv = d_lvl2_bcat_raw_csv.sort_values('_date_sort').drop(columns=['_date_sort']).reset_index(drop=True)

csv_path = f"{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.csv"
d_lvl2_bcat_raw_csv.to_csv(csv_path, index=False)

print(f"Fetched {len(d_lvl2_bcat_raw_csv):,} events")
print(f"Saved to: {csv_path}")
d_lvl2_bcat_raw_csv.head(3)

Fetching data from Bellingcat API...
Raw JSON saved to: data/raw/level_2/ukr-civharm-2026-07-29.json
Fetched 2,517 events
Saved to: data/raw/level_2/ukr-civharm-2026-07-29.csv


,id,date,latitude,longitude,location,description,sources,associations
0,CIV0003,02/24/2022,50.470772,30.528098,,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/TreyYingst/status/14967937...,
1,CIV0098,02/24/2022,49.212119,38.905921,,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste..."
2,CIV0013,02/24/2022,48.055395,37.778300,,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System..."


In [8]:
# Dupe raw for processing
d_lvl2_bcat = d_lvl2_bcat_raw_csv.copy()

# Add ascending numerical index
d_lvl2_bcat['index'] = range(len(d_lvl2_bcat))
d_lvl2_bcat = d_lvl2_bcat.set_index('index')

# Drop imprecise OS location col
d_lvl2_bcat = d_lvl2_bcat.drop(
    'location', 
    axis = 1, 
    errors = 'ignore',
    )

# Restrict to obs on or before ULS start date
d_lvl2_bcat['date'] = pd.to_datetime(
    d_lvl2_bcat['date'], 
    format = DATE_FORMAT_INPUT,
    errors = 'coerce',    
    )

uls_startdate = pd.to_datetime(SURVEY_START_DATE)
d_lvl2_bcat = d_lvl2_bcat[d_lvl2_bcat['date'] <= uls_startdate]

# Inspect & verify
d_lvl2_bcat.shape
d_lvl2_bcat.info()
d_lvl2_bcat.head(2)
d_lvl2_bcat.tail(2)

(2446, 7)

<class 'pandas.core.frame.DataFrame'>
Index: 2446 entries, 0 to 2445
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            2446 non-null   object        
 1   date          2446 non-null   datetime64[ns]
 2   latitude      2446 non-null   float64       
 3   longitude     2446 non-null   float64       
 4   description   2446 non-null   object        
 5   sources       2446 non-null   object        
 6   associations  2446 non-null   object        
dtypes: datetime64[ns](1), float64(2), object(4)
memory usage: 152.9+ KB


,id,date,latitude,longitude,description,sources,associations
index,,,,,,,
0,CIV0003,2022-02-24,50.470772,30.528098,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/TreyYingst/status/14967937...,
1,CIV0098,2022-02-24,49.212119,38.905921,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste..."


,id,date,latitude,longitude,description,sources,associations
index,,,,,,,
2444,S23173,2025-04-07,48.507810,37.747111,At least one person killed and two injured inc...,"https://t.me/astrapress/78387,https://t.me/ast...","Type of area affected=Residential,Type of area..."
2445,W4DR1R,2025-04-08,50.775975,35.251465,Heavily damaged residential buildings followin...,"https://t.me/suspilnesumy/32400,https://t.me/s...",Type of area affected=Residential


In [9]:
### NOTE 7/28: verified: download + API fetch produce identical data structures; deprecating download

#d.to_csv('d_api.csv')

In [ ]:
# Dummy code: area type affected & weapon system

### Creates binary indicators for all `associations` values

# === TYPE OF AREA AFFECTED ===
area_types = {
    'a00': 'Administrative',           
    'a01': 'Commercial',
    'a02': 'Cultural',
    'a03': 'Food/Food Infrastructure',
    'a04': 'Healthcare',
    'a05': 'Humanitarian',
    'a06': 'Industrial',
    'a07': 'Military',
    'a08': 'Religious',
    'a09': 'Residential',
    'a10': 'Roads/Highways/Transport',
    'a11': 'School or childcare',
    'undefined': 'Undefined',
    }

for var, label in area_types.items():
    d_lvl2_bcat[var] = d_lvl2_bcat['associations'].str.contains(
        rf'Type of area affected={re.escape(label)}',
        case=False,
        na=False,
        regex=True,
    ).astype(int)

# === WEAPON SYSTEM ===
weapon_systems = {
    'w00': 'Air strike',
    'w01': 'Anti-air missile',
    'w02': 'Ballistic missile',
    'w03': 'Cluster munitions',
    'w04': 'Cruise missile',
    'w05': 'HE artillery inc mortars',
    'w06': 'HE rocket artillery',
    'w07': 'HE tube artillery',
    'w08': 'Incendiary munitions',
    'w09': 'Land mines',
    'w10': 'Loitering munition',
    'w11': 'Small arms',
    'w12': 'Thermobaric munition',
    'w13': 'Vehicle mounted weapon',
    'unknown': 'Unknown',
    'none': 'None',
    }

for var, label in weapon_systems.items():
    d_lvl2_bcat[var] = d_lvl2_bcat['associations'].str.contains(
        rf'Weapon System={re.escape(label)}',
        case=False,
        na=False,
        regex=True,
    ).astype(int)

# Verify counts

print("=== TYPE OF AREA AFFECTED ===")
for var, label in area_types.items():
    print(f"  {var} ({label}): {d_lvl2_bcat[var].sum()}")

print("\n=== WEAPON SYSTEM ===")
for var, label in weapon_systems.items():
    print(f"  {var} ({label}): {d_lvl2_bcat[var].sum()}")

=== TYPE OF AREA AFFECTED ===
  a00 (Administrative): 123
  a01 (Commercial): 395
  a02 (Cultural): 105
  a03 (Food/Food Infrastructure): 52
  a04 (Healthcare): 139
  a05 (Humanitarian): 34
  a06 (Industrial): 181
  a07 (Military): 7
  a08 (Religious): 65
  a09 (Residential): 1094
  a10 (Roads/Highways/Transport): 231
  a11 (School or childcare): 326
  undefined (Undefined): 71

=== WEAPON SYSTEM ===
  w00 (Air strike): 68
  w01 (Anti-air missile): 40
  w02 (Ballistic missile): 47
  w03 (Cluster munitions): 117
  w04 (Cruise missile): 82
  w05 (HE artillery inc mortars): 13
  w06 (HE rocket artillery): 62
  w07 (HE tube artillery): 6
  w08 (Incendiary munitions): 16
  w09 (Land mines): 6
  w10 (Loitering munition): 111
  w12 (Small arms): 31
  w13 (Thermobaric munition): 8
  w14 (Vehicle mounted weapon): 16
  unknown (Unknown): 1300
  none (None): 1


In [11]:
#############################################################################################
d_lvl2_bcat.head(5)
#############################################################################################

,id,date,latitude,longitude,description,sources,associations,a00,a01,a02,a03,a04,a05,a06,a07,a08,a09,a10,a11,undefined,w00,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w12,w13,w14,unknown,none
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,CIV0003,2022-02-24,50.470772,30.528098,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/TreyYingst/status/14967937...,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,CIV0098,2022-02-24,49.212119,38.905921,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste...",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,CIV0013,2022-02-24,48.055395,37.778300,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System...",0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,CIV0004,2022-02-24,47.775609,37.239673,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/N_Waters89/status/14968566...,"Type of area affected=Healthcare,Weapon System...",0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,CIV0011,2022-02-24,50.308310,34.880702,Civilian buildings damaged and destroyed by sh...,https://t.me/nexta_live/16696,"Type of area affected=Commercial,Weapon System...",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


### 3. Import + transform: Uppsala Conflict Data Program
Imports, cleans, describes level-2 aggregate conflict data. Acquired via .csv: https://ucdp.uu.se/downloads/

In [12]:
# Fetch UCDP Georeferenced Event Dataset Global

### docs: https://ucdp.uu.se/downloads/index.html
### codebook: https://ucdp.uu.se/downloads/ged/ged261.pdf

def fetch_ucdp_ged(url, output_dir):
    """
    Downloads and extracts UCDP GED CSV from zip archive.
    Returns path to extracted CSV file.
    """
    zip_filename = url.split('/')[-1]
    zip_path = f"{output_dir}/{zip_filename}"
    
    # Download zip file
    print(f"Downloading {zip_filename}...")
    urllib.request.urlretrieve(url, zip_path)
    print(f"Downloaded to: {zip_path}")
    
    # Extract CSV from zip
    print(f"Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        csv_files = [f for f in z.namelist() if f.endswith('.csv')]
        if not csv_files:
            raise ValueError("No CSV file found in archive")
        
        csv_filename = csv_files[0]
        z.extract(csv_filename, output_dir)
        csv_path = f"{output_dir}/{csv_filename}"
    
    # Remove zip file after extraction
    os.remove(zip_path)
    print(f"Extracted to: {csv_path}")
    
    return csv_path

# Fetch UCDP GED
ucdp_csv_path = fetch_ucdp_ged(UCDP_GED_URL, DATA_LVL2)

# Load and preview
d_lvl2_ucdp_raw = pd.read_csv(ucdp_csv_path, low_memory=False)
print(f"UCDP GED loaded: {len(d_lvl2_ucdp_raw):,} events (global)")
print(f"Columns: {d_lvl2_ucdp_raw.columns.tolist()}")
d_lvl2_ucdp_raw.head(3)

Downloaded to: data/raw/level_2/ged261-csv.zip
Extracting...
Extracted to: data/raw/level_2/GEDEvent_v26_1.csv
UCDP GED loaded: 417,968 events (global)
Columns: ['id', 'relid', 'year', 'active_year', 'code_status', 'type_of_violence', 'conflict_dset_id', 'conflict_new_id', 'conflict_name', 'dyad_dset_id', 'dyad_new_id', 'dyad_name', 'side_a_dset_id', 'side_a_new_id', 'side_a', 'side_b_dset_id', 'side_b_new_id', 'side_b', 'number_of_sources', 'source_article', 'source_office', 'source_date', 'source_headline', 'source_original', 'where_prec', 'where_coordinates', 'where_description', 'adm_1', 'adm_2', 'latitude', 'longitude', 'geom_wkt', 'priogrid_gid', 'country', 'country_id', 'region', 'event_clarity', 'date_prec', 'date_start', 'date_end', 'deaths_a', 'deaths_b', 'deaths_civilians', 'deaths_unknown', 'best', 'high', 'low', 'gwnoa', 'gwnob']


,id,relid,year,active_year,code_status,type_of_violence,conflict_dset_id,conflict_new_id,conflict_name,dyad_dset_id,dyad_new_id,dyad_name,side_a_dset_id,side_a_new_id,side_a,side_b_dset_id,side_b_new_id,side_b,number_of_sources,source_article,source_office,source_date,source_headline,source_original,where_prec,where_coordinates,where_description,adm_1,adm_2,latitude,longitude,geom_wkt,priogrid_gid,country,country_id,region,event_clarity,date_prec,date_start,date_end,deaths_a,deaths_b,deaths_civilians,deaths_unknown,best,high,low,gwnoa,gwnob
0,1568,ALG-1992-1-1-6,1992,True,Clear,1,386.0,386,Algeria: Government,828.0,828,Government of Algeria - AIS,109.0,109,Government of Algeria,537.0,537,AIS,-1,Reuters 3/19/1992 ALGERIAN SECURITY WARNS OF K...,NaN,NaN,NaN,NaN,1,Medea town,"Medea town, Medea district, Medea province",Medea province,Medea commune,36.264169,2.753926,POINT (2.753926 36.264169),181806,Algeria,615,Africa,1,1,1992-03-17 00:00:00.000,1992-03-17 00:00:00.000,1,0,1,0,2,2,2,615,NaN
1,1572,ALG-1992-1-1-109,1992,True,Clear,1,386.0,386,Algeria: Government,828.0,828,Government of Algeria - AIS,109.0,109,Government of Algeria,537.0,537,AIS,-1,Reuters 12/21/1992 Gunmen kill Algerian gendar...,NaN,NaN,NaN,NaN,1,Ksar El-Boukhari town,"Ksar El-Boukhari town, Ksar El-Boukhari distri...",Medea province,Ksar El-Boukhari commune,35.888887,2.749048,POINT (2.749048 35.888887),181086,Algeria,615,Africa,1,1,1992-12-20 00:00:00.000,1992-12-20 00:00:00.000,1,0,1,0,2,2,2,615,NaN
2,1593,ALG-1992-1-1-66,1992,True,Clear,1,386.0,386,Algeria: Government,828.0,828,Government of Algeria - AIS,109.0,109,Government of Algeria,537.0,537,AIS,-1,Reuters 9/28/1992 Two Algerian officers killed...,NaN,NaN,NaN,NaN,4,Blida province,Blida province,Blida province,NaN,36.583333,3.000000,POINT (3 36.5833333),182527,Algeria,615,Africa,1,1,1992-09-27 00:00:00.000,1992-09-27 00:00:00.000,2,0,0,0,2,2,2,615,NaN


In [13]:
# Dupe raw for processing
d_lvl2_ucdp = d_lvl2_ucdp_raw.copy()

# Filter to Russia-Ukraine conflict only
d_lvl2_ucdp = d_lvl2_ucdp[d_lvl2_ucdp['conflict_name'] == 'Russia - Ukraine']
print(f"Filtered to Russia-Ukraine: {len(d_lvl2_ucdp):,} events")

# Convert date_start to datetime
d_lvl2_ucdp['date_start'] = pd.to_datetime(d_lvl2_ucdp['date_start'], errors='coerce')

# Filter to invasion start (2022-02-24) through ULS survey start
invasion_start = pd.to_datetime('2022-02-24')
uls_startdate = pd.to_datetime(SURVEY_START_DATE)

d_lvl2_ucdp = d_lvl2_ucdp[
    (d_lvl2_ucdp['date_start'] >= invasion_start) & 
    (d_lvl2_ucdp['date_start'] <= uls_startdate)
]
print(f"Filtered to {invasion_start.date()} – {uls_startdate.date()}: {len(d_lvl2_ucdp):,} events")

# Sort by date ascending
d_lvl2_ucdp = d_lvl2_ucdp.sort_values('date_start').reset_index(drop=True)

# Inspect & verify
d_lvl2_ucdp.shape
d_lvl2_ucdp.info()
d_lvl2_ucdp.head(3)

Filtered to Russia-Ukraine: 37,892 events
Filtered to 2022-02-24 – 2025-04-08: 31,792 events


(31792, 49)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31792 entries, 0 to 31791
Data columns (total 49 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 31792 non-null  int64         
 1   relid              31792 non-null  object        
 2   year               31792 non-null  int64         
 3   active_year        31792 non-null  bool          
 4   code_status        31792 non-null  object        
 5   type_of_violence   31792 non-null  int64         
 6   conflict_dset_id   31792 non-null  float64       
 7   conflict_new_id    31792 non-null  int64         
 8   conflict_name      31792 non-null  object        
 9   dyad_dset_id       31792 non-null  float64       
 10  dyad_new_id        31792 non-null  int64         
 11  dyad_name          31792 non-null  object        
 12  side_a_dset_id     31792 non-null  float64       
 13  side_a_new_id      31792 non-null  int64         
 14  side_a

,id,relid,year,active_year,code_status,type_of_violence,conflict_dset_id,conflict_new_id,conflict_name,dyad_dset_id,dyad_new_id,dyad_name,side_a_dset_id,side_a_new_id,side_a,side_b_dset_id,side_b_new_id,side_b,number_of_sources,source_article,source_office,source_date,source_headline,source_original,where_prec,where_coordinates,where_description,adm_1,adm_2,latitude,longitude,geom_wkt,priogrid_gid,country,country_id,region,event_clarity,date_prec,date_start,date_end,deaths_a,deaths_b,deaths_civilians,deaths_unknown,best,high,low,gwnoa,gwnob
0,441052,RUS-2022-1-14117-905,2022,True,Clear,1,13243.0,13243,Russia - Ukraine,14117.0,14117,Government of Russia (Soviet Union) - Governme...,57.0,57,Government of Russia (Soviet Union),61.0,61,Government of Ukraine,1,"""Ukrinform: news,2022-06-14,In a village in th...",Ukrinform: news,2022-06-14,"In a village in the Kharkiv region, the enemy ...",Ukrinform,1,Zolochiv village,the Zolochiv community,Kharkiv oblast,Bohodukhiv raion,50.272064,35.980068,POINT (35.980068 50.272064),202032,Ukraine,369,Europe,2,5,2022-02-24,2022-06-14 00:00:00.000,0,0,15,0,15,15,15,365,369.0
1,512726,UKR-2022-1-14117-5391,2022,True,Clear,1,13243.0,13243,Russia - Ukraine,14117.0,14117,Government of Russia (Soviet Union) - Governme...,57.0,57,Government of Russia (Soviet Union),61.0,61,Government of Ukraine,2,"""UALosses,2025-01-17,Ukraine's losses in the w...",UALosses;UALosses,2025-01-17;2026-02-03,Ukraine's losses in the war 24.02.2022 - 28.02...,UALosses,3,Mariupol raion,Battle of Mariupol,Donetsk oblast,Mariupol raion,47.145783,37.584772,POINT (37.584772 47.145783),197716,Ukraine,369,Europe,2,2,2022-02-24,2022-02-28 00:00:00.000,0,18,0,0,18,18,18,365,369.0
2,433313,RUS-2022-1-14117-100,2022,True,Clear,1,13243.0,13243,Russia - Ukraine,14117.0,14117,Government of Russia (Soviet Union) - Governme...,57.0,57,Government of Russia (Soviet Union),61.0,61,Government of Ukraine,1,"""Deccan Herald,2022-03-11,In a Mykolaiv morgue...",Deccan Herald,2022-03-11,"In a Mykolaiv morgue, corpses pile up in the snow",morgue,2,Mykolaiv town,NaN,Mykolayiv oblast,Mykolayiv raion,46.959201,31.987586,POINT (31.987586 46.959201),196984,Ukraine,369,Europe,2,4,2022-02-24,2022-03-11 00:00:00.000,0,27,14,0,41,41,41,365,369.0


In [14]:
# Data condensation: keep only relevant columns
cols_keep = [
    'conflict_name',
    'source_article',
    'source_office',
    'source_headline',
    'source_original',
    'where_prec',
    'where_description',
    'adm_1',
    'adm_2',
    'latitude',
    'longitude',
    'geom_wkt',
    'priogrid_gid',
    'country',
    'event_clarity',
    'date_prec',
    'date_start',
    'date_end',
    'deaths_a',
    'deaths_b',
    'deaths_civilians',
    'deaths_unknown',
    'best',
    'high',
    ]

d_lvl2_ucdp = d_lvl2_ucdp[cols_keep]

print(f"Condensed to {len(cols_keep)} columns")
d_lvl2_ucdp.info()
d_lvl2_ucdp.head(3)

Condensed to 24 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31792 entries, 0 to 31791
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   conflict_name      31792 non-null  object        
 1   source_article     31792 non-null  object        
 2   source_office      31792 non-null  object        
 3   source_headline    31788 non-null  object        
 4   source_original    31525 non-null  object        
 5   where_prec         31792 non-null  int64         
 6   where_description  30940 non-null  object        
 7   adm_1              29813 non-null  object        
 8   adm_2              25534 non-null  object        
 9   latitude           31792 non-null  float64       
 10  longitude          31792 non-null  float64       
 11  geom_wkt           31792 non-null  object        
 12  priogrid_gid       31792 non-null  int64         
 13  country            31792 non-null  ob

,conflict_name,source_article,source_office,source_headline,source_original,where_prec,where_description,adm_1,adm_2,latitude,longitude,geom_wkt,priogrid_gid,country,event_clarity,date_prec,date_start,date_end,deaths_a,deaths_b,deaths_civilians,deaths_unknown,best,high
0,Russia - Ukraine,"""Ukrinform: news,2022-06-14,In a village in th...",Ukrinform: news,"In a village in the Kharkiv region, the enemy ...",Ukrinform,1,the Zolochiv community,Kharkiv oblast,Bohodukhiv raion,50.272064,35.980068,POINT (35.980068 50.272064),202032,Ukraine,2,5,2022-02-24,2022-06-14 00:00:00.000,0,0,15,0,15,15
1,Russia - Ukraine,"""UALosses,2025-01-17,Ukraine's losses in the w...",UALosses;UALosses,Ukraine's losses in the war 24.02.2022 - 28.02...,UALosses,3,Battle of Mariupol,Donetsk oblast,Mariupol raion,47.145783,37.584772,POINT (37.584772 47.145783),197716,Ukraine,2,2,2022-02-24,2022-02-28 00:00:00.000,0,18,0,0,18,18
2,Russia - Ukraine,"""Deccan Herald,2022-03-11,In a Mykolaiv morgue...",Deccan Herald,"In a Mykolaiv morgue, corpses pile up in the snow",morgue,2,NaN,Mykolayiv oblast,Mykolayiv raion,46.959201,31.987586,POINT (31.987586 46.959201),196984,Ukraine,2,4,2022-02-24,2022-03-11 00:00:00.000,0,27,14,0,41,41


#### a. Reverse geocode: latitude / longitude $\rightarrow$ UA postcode
Reverse geocodes event coordinates to Ukrainian postcodes via Nominatim API.

In [15]:
#############################################################################################
#############################################################################################
# Restrict to n = 100 for geocoding tests
d_lvl2_bcat = d_lvl2_bcat.iloc[:100]
d_lvl2_bcat.info()
#############################################################################################
#############################################################################################

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 36 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            100 non-null    object        
 1   date          100 non-null    datetime64[ns]
 2   latitude      100 non-null    float64       
 3   longitude     100 non-null    float64       
 4   description   100 non-null    object        
 5   sources       100 non-null    object        
 6   associations  100 non-null    object        
 7   a00           100 non-null    int64         
 8   a01           100 non-null    int64         
 9   a02           100 non-null    int64         
 10  a03           100 non-null    int64         
 11  a04           100 non-null    int64         
 12  a05           100 non-null    int64         
 13  a06           100 non-null    int64         
 14  a07           100 non-null    int64         
 15  a08           100 non-null    int64         
 

In [16]:
# Convert latitude/longitude coordinates → Ukrainian postcodes via Nominatim API
geolocator = Nominatim(user_agent = NOMINATIM_USER_AGENT)

def get_postcode(lat, lon):
    """
    Reverse geocode latitude/longitude to get postcode.
    Returns None if postcode not found.
    """
    try:
        location = geolocator.reverse(f"{lat}, {lon}", language = 'en')
        if location and location.raw.get('address'):
            postcode = location.raw['address'].get('postcode')
            return postcode
        return None
    except Exception as e:
        print(f"Error geocoding ({lat}, {lon}): {e}")
        return None

# Apply geocoding to each row with rate-limited delay
postcodes = []

for idx, row in tqdm(d_lvl2_bcat.iterrows(), total=len(d_lvl2_bcat), desc="Geocoding postcodes"):
    lat = row['latitude']
    lon = row['longitude']
    
    postcode = get_postcode(lat, lon)
    postcodes.append(postcode)
    
    sleep(NOMINATIM_DELAY_SEC)

d_lvl2_bcat['postcode'] = postcodes

print(f"\nGeocoding complete!")
print(f"Postcodes found: {d_lvl2_bcat['postcode'].notna().sum()}/{len(d_lvl2_bcat)}")
print(f"\nSample results:")
print(d_lvl2_bcat[['latitude', 'longitude', 'postcode']].head(10))

Geocoding postcodes:   0%|          | 0/100 [00:00<?, ?it/s]


Geocoding complete!
Postcodes found: 99/100

Sample results:
        latitude  longitude postcode
index                               
0      50.470772  30.528098    04070
1      49.212119  38.905921    92760
2      48.055395  37.778300    83054
3      47.775609  37.239673    85670
4      50.308310  34.880702    42700
5      47.117099  37.684831    87503
6      47.120906  37.680990    87503
7      48.748938  30.218889    20305
8      46.227914  34.642854    75565
9      47.379841  37.756229    87100


In [17]:
# Enumerate unique postcodes in d_lvl2_bcat
def count_unique_postcodes(df, col = 'postcode'):
    """
    Returns count of unique non-null values in specified column.
    """
    unique_vals = df[col].dropna().unique()
    return len(unique_vals)

n_unique = count_unique_postcodes(d_lvl2_bcat)
print(f"Unique postcodes in d_lvl2_bcat: {n_unique}")

Unique postcodes in d_lvl2_bcat: 75


In [18]:
# Extract first 2 digits of postcode as `admin_unit` id
d_lvl2_bcat['admin_unit'] = d_lvl2_bcat['postcode'].astype(str).str[:2]

# Replace 'na' (from NaN conversion) with actual NaN
d_lvl2_bcat.loc[d_lvl2_bcat['admin_unit'] == 'na', 'admin_unit'] = np.nan

# Enumerate unique admin units
n_unique_admin = count_unique_postcodes(
    d_lvl2_bcat, 
    col = 'admin_unit',
    )
print(f"Unique admin units in d_lvl2_bcat: {n_unique_admin}")
print(f"\nAdmin unit distribution:")
d_lvl2_bcat['admin_unit'].value_counts().sort_index()

Unique admin units in d_lvl2_bcat: 27

Admin unit distribution:


admin_unit
01     1
02     1
03     1
04     5
07     2
08     9
14     4
15     1
20     1
34     1
42     4
54     1
61    29
62     2
63     2
64     1
72     1
73     2
74     2
75     1
83     6
84     3
85     5
87     6
92     4
93     4
No     1
Name: count, dtype: int64

In [19]:
# Map `admin_unit` → `oblast`
### Uses ADMIN_UNIT_TO_OBLAST from mappings.py
### Source: Ukrposhta postal code system: https://en.wikipedia.org/wiki/Postal_codes_in_Ukraine

d_lvl2_bcat['oblast'] = d_lvl2_bcat['admin_unit'].map(ADMIN_UNIT_TO_OBLAST)

# Verify mapping
print(f"Mapped oblasts: {d_lvl2_bcat['oblast'].notna().sum()}/{len(d_lvl2_bcat)}")
print(f"Unmapped admin units: {d_lvl2_bcat[d_lvl2_bcat['oblast'].isna()]['admin_unit'].unique()}")
print(f"\nOblast distribution:")
d_lvl2_bcat['oblast'].value_counts()

Mapped oblasts: 99/100
Unmapped admin units: ['No']

Oblast distribution:


oblast
Kharkiv Oblast         34
Donetsk Oblast         20
Kyiv Oblast            11
Kyiv                    8
Luhansk Oblast          8
Kherson Oblast          5
Chernihiv Oblast        5
Sumy Oblast             4
Cherkasy Oblast         1
Zaporizhzhia Oblast     1
Rivne Oblast            1
Mykolaiv Oblast         1
Name: count, dtype: int64

#### PRELIM: Encode _raions_ two ways
**Note:** Ukrposhta postcode lookup will not return Russian-occupied districts. These are denoted `<Rus-occupied>` in `raion_postcode`:<br>

|Prefix|Region|$n$ raions|
|------|------|------------------|
|83xxx|Donetsk city|0 (`<Rus-occupied>`)|
|91xxx|Luhansk city|0 (`<Rus-occupied>`)|
|94xxx|Luhansk oblast|0 (`<Rus-occupied>`)|
|95-99|Crimea/Sevastopol|0 (`<Rus-occupied>`)|

In [20]:
# Map latitude/longitude coordinates → Ukrainian raions via Nominatim API
### Source: ukraine_raion_lookup.py (option 4)
### For Ukraine, Nominatim returns raion in the "district" field
### Rate-limited: 1 req/sec on public instance; not suitable for bulk geocoding

def raion_from_point_nominatim(lat, lon, user_agent, email=None):
    """
    Reverse geocode lat/lon to Ukrainian raion via OSM Nominatim.
    Returns raion name from 'district' field, or None if not found.
    """
    params = {
        'lat': lat,
        'lon': lon,
        'format': 'jsonv2',
        'addressdetails': 1,
    }
    headers = {'User-Agent': user_agent}
    if email:
        params['email'] = email
    
    try:
        resp = requests.get(
            'https://nominatim.openstreetmap.org/reverse',
            params=params,
            headers=headers,
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
        sleep(NOMINATIM_DELAY_SEC)  # respect rate limit
        return data.get('address', {}).get('district')
    except Exception as e:
        print(f"Error geocoding ({lat}, {lon}): {e}")
        return None

# Apply to dataframe with progress bar
raions_nominatim = []

for idx, row in tqdm(d_lvl2_bcat.iterrows(), total=len(d_lvl2_bcat), desc="Geocoding raions"):
    raion = raion_from_point_nominatim(
        row['latitude'], 
        row['longitude'], 
        user_agent=NOMINATIM_USER_AGENT,
    )
    raions_nominatim.append(raion)

d_lvl2_bcat['raion_nominatim_ua'] = raions_nominatim

print(f"\nRaion geocoding complete!")
print(f"Raions found: {d_lvl2_bcat['raion_nominatim_ua'].notna().sum()}/{len(d_lvl2_bcat)}")
print(f"\nSample results:")
d_lvl2_bcat[['latitude', 'longitude', 'raion_nominatim_ua']].head(10)

Geocoding raions:   0%|          | 0/100 [00:00<?, ?it/s]


Raion geocoding complete!
Raions found: 92/100

Sample results:


,latitude,longitude,raion_nominatim_ua
index,,,
0,50.470772,30.528098,None
1,49.212119,38.905921,Старобільський район
2,48.055395,37.778300,Донецький район
3,47.775609,37.239673,Волноваський район
4,50.308310,34.880702,Охтирський район
5,47.117099,37.684831,Маріупольський район
6,47.120906,37.680990,Маріупольський район
7,48.748938,30.218889,Уманський район
8,46.227914,34.642854,Генічеський район


In [21]:
# Map Ukrainian raion names → English translations
### Uses RAION_UA_TO_EN from mappings.py
### Source: https://en.wikipedia.org/wiki/Raions_of_Ukraine (post-2020 reform: 136 raions)

d_lvl2_bcat['raion_nominatim_en'] = d_lvl2_bcat['raion_nominatim_ua'].map(RAION_UA_TO_EN)

# Report coverage
mapped = d_lvl2_bcat['raion_nominatim_en'].notna().sum()
total_with_ua = d_lvl2_bcat['raion_nominatim_ua'].notna().sum()
print(f"Mapped to English: {mapped}/{total_with_ua}")

# Show any unmapped Ukrainian raions for dictionary updates
unmapped = d_lvl2_bcat[d_lvl2_bcat['raion_nominatim_ua'].notna() & d_lvl2_bcat['raion_nominatim_en'].isna()]['raion_nominatim_ua'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped raions (add to mappings.py):")
    for r in unmapped:
        print(f"    '{r}': '',")

print(f"\nSample results:")
d_lvl2_bcat[['raion_nominatim_ua', 'raion_nominatim_en']].head(10)

Mapped to English: 88/92

Unmapped raions (add to mappings.py):
    'Сіверськодонецький район': '',

Sample results:


,raion_nominatim_ua,raion_nominatim_en
index,,
0,None,NaN
1,Старобільський район,Starobilsk Raion
2,Донецький район,Donetsk Raion
3,Волноваський район,Volnovakha Raion
4,Охтирський район,Okhtyrka Raion
5,Маріупольський район,Mariupol Raion
6,Маріупольський район,Mariupol Raion
7,Уманський район,Uman Raion
8,Генічеський район,Henichesk Raion


In [22]:
# Extract postcode directory from .7z archive
import py7zr

POSTCODE_7Z_PATH = f'{DATA_RAW}/{POSTCODE_7Z}'

# Extract if .7z exists
if os.path.exists(POSTCODE_7Z_PATH):
    # Check if already extracted by looking for any CSV with Ukrainian postal keywords
    existing_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv') and 'індекс' in f.lower()]
    
    if not existing_csvs:
        print(f"Extracting {POSTCODE_7Z_PATH}...")
        with py7zr.SevenZipFile(POSTCODE_7Z_PATH, mode='r') as archive:
            archive.extractall(path=DATA_RAW)
        print(f"Extracted to: {DATA_RAW}/")
        existing_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv') and 'індекс' in f.lower()]
    
    # Set path to the extracted CSV
    if existing_csvs:
        POSTCODE_DIR_PATH = f'{DATA_RAW}/{existing_csvs[0]}'
        print(f"Postcode directory: {POSTCODE_DIR_PATH}")
    else:
        # Fallback: find any newly created CSV
        all_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv')]
        print(f"CSV files found: {all_csvs}")
        if all_csvs:
            POSTCODE_DIR_PATH = f'{DATA_RAW}/{all_csvs[0]}'
            print(f"Using: {POSTCODE_DIR_PATH}")
else:
    print(f"Archive not found: {POSTCODE_7Z_PATH}")
    print("Download from: https://data.gov.ua/dataset/post-index-and-braches")
    POSTCODE_DIR_PATH = None

Postcode directory: data/raw/Звіт для МІУ. Перелік поштових індексів та відділень_08.08.2025.csv


In [23]:
# Map postcodes → Ukrainian raions via Ministry of Community and Territorial Development of Ukraine open data
### Source: ukraine_raion_lookup.py (option 3)
### Data: https://data.gov.ua/dataset/post-index-and-braches
### Note: Column headers are in Ukrainian and may vary between releases; inspect after download

def load_postcode_directory(csv_path):
    """
    Load the data.gov.ua "post-index-and-braches" CSV.
    Tries multiple encodings common for Ukrainian government data.
    Uses semicolon delimiter (European CSV format).
    """
    encodings = ['cp1251', 'windows-1251', 'utf-8', 'iso-8859-5', 'utf-16']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(
                csv_path, 
                encoding=encoding, 
                dtype=str,
                sep=';',            ### European .CSV uses semicolon
                on_bad_lines='skip' ### Skips malformed rows
            )
            print(f"Successfully loaded with encoding: {encoding}")
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    raise ValueError(f"Could not decode {csv_path} with any known encoding")

def raion_from_postcode(postcode, directory, postcode_col, raion_col):
    """
    Direct table lookup for postcode → raion.
    Ukrainian postal codes do NOT cleanly encode raion in digit positions;
    use this authoritative directory rather than parsing the string.
    """
    row = directory.loc[directory[postcode_col] == str(postcode)]
    if row.empty:
        return None
    return row.iloc[0][raion_col]

# Load directory and inspect columns
try:
    postcode_dir = load_postcode_directory(POSTCODE_DIR_PATH)
    print(f"Postcode directory loaded: {len(postcode_dir):,} entries")
    print(f"Columns: {postcode_dir.columns.tolist()}")
    
    # Use English column names from the file
    # Adjust these if your file has different column names
    POSTCODE_COL = 'Postindex VPZ'    ### postcode column
    RAION_COL = 'Distinct (Rayon)'    ### raion column (note: "Distinct" is likely a typo for "District")
    
    # Verify columns exist
    if POSTCODE_COL not in postcode_dir.columns or RAION_COL not in postcode_dir.columns:
        print(f"\nWARNING: Expected columns not found!")
        print(f"Looking for: '{POSTCODE_COL}', '{RAION_COL}'")
        print(f"Available: {postcode_dir.columns.tolist()}")
    else:
        # Apply lookup to dataframe
        d_lvl2_bcat['raion_postcode'] = d_lvl2_bcat['postcode'].apply(
            lambda pc: raion_from_postcode(pc, postcode_dir, POSTCODE_COL, RAION_COL)
        )
        
        # Replace None with <Rus-occupied> (postcodes in occupied territories not in Ukrposhta directory)
        d_lvl2_bcat['raion_postcode'] = d_lvl2_bcat['raion_postcode'].fillna('<Rus-occupied>')
        
        print(f"\nRaions found via postcode: {(d_lvl2_bcat['raion_postcode'] != '<Rus-occupied>').sum()}/{len(d_lvl2_bcat)}")
        print(f"Rus-occupied: {(d_lvl2_bcat['raion_postcode'] == '<Rus-occupied>').sum()}/{len(d_lvl2_bcat)}")
        print(f"\nSample results:")
        d_lvl2_bcat[['postcode', 'raion_postcode']].head(10)
    
except FileNotFoundError:
    print(f"Postcode directory not found at: {POSTCODE_DIR_PATH}")
    print("Download from: https://data.gov.ua/dataset/post-index-and-braches")
    print("Save as 'postindex.7z' in data/raw/ and run extraction cell above")

Successfully loaded with encoding: cp1251
Postcode directory loaded: 320,249 entries
Columns: ['Назва області', 'Назва району', 'Назва населеного пункту (повна)', 'Поштовий індекс населеного пункту', 'Назва вулиці', 'Номер будинку', "Назва відділення зв'язку", "Поштовий індекс відділення зв'язку (ВПЗ)", 'Region (Oblast)', 'Distinct (Rayon)', 'Locality', 'Postindex Locality', 'Street', 'House_numbers', 'Post office', 'Postindex VPZ']

Raions found via postcode: 72/100
Rus-occupied: 28/100

Sample results:


,postcode,raion_postcode
index,,
0,04070,Kyiv
1,92760,Starobilskyi
2,83054,<Rus-occupied>
3,85670,Volnovaskyi
4,42700,<Rus-occupied>
5,87503,Mariupolskyi
6,87503,Mariupolskyi
7,20305,Umanskyi
8,75565,<Rus-occupied>


In [24]:
#############################################################################################
# DEBUG: Investigate postcode lookup misses

test_postcode = '83054'
print(f"Investigating postcode: {test_postcode}\n")

# Check both postcode columns in directory
postcode_cols = ['Postindex VPZ', 'Postindex Locality']
for col in postcode_cols:
    if col in postcode_dir.columns:
        exact = postcode_dir[postcode_dir[col] == test_postcode]
        partial = postcode_dir[postcode_dir[col].str.contains(test_postcode, na=False)]
        print(f"'{col}':")
        print(f"  Exact match: {len(exact)} rows")
        print(f"  Partial match: {len(partial)} rows")
        print(f"  Sample values: {postcode_dir[col].dropna().unique()[:10].tolist()}\n")

# Check what postcodes ARE in directory for Donetsk oblast (83-87)
donetsk_postcodes = postcode_dir[postcode_dir['Postindex VPZ'].str.startswith('83', na=False)]
print(f"Donetsk (83xxx) postcodes in directory: {len(donetsk_postcodes)}")
if len(donetsk_postcodes) > 0:
    print(f"Sample: {donetsk_postcodes['Postindex VPZ'].unique()[:10].tolist()}")

# Check Luhansk (91-94)
luhansk_postcodes = postcode_dir[postcode_dir['Postindex VPZ'].str.startswith('92', na=False)]
print(f"\nLuhansk (92xxx) postcodes in directory: {len(luhansk_postcodes)}")
if len(luhansk_postcodes) > 0:
    print(f"Sample: {luhansk_postcodes['Postindex VPZ'].unique()[:10].tolist()}")

# Summary: which oblasts are missing?
print(f"\nOblast prefixes in directory:")
postcode_dir['prefix'] = postcode_dir['Postindex VPZ'].str[:2]
print(postcode_dir['prefix'].value_counts().sort_index())
#############################################################################################

Investigating postcode: 83054

'Postindex VPZ':
  Exact match: 0 rows
  Partial match: 0 rows
  Sample values: ['21001', '21002', '21003', '21004', '21005', '21006', '21007', '21008', '21009', '21010']

'Postindex Locality':
  Exact match: 0 rows
  Partial match: 0 rows
  Sample values: ['21001', '21002', '21003', '21004', '21005', '21006', '21007', '21008', '21009', '21010']

Donetsk (83xxx) postcodes in directory: 0

Luhansk (92xxx) postcodes in directory: 2122
Sample: ['92100', '92110', '92111', '92112', '92114', '92115', '92120', '92121', '92123', '92124']

Oblast prefixes in directory:
prefix
01     299
02     691
03    1112
04     530
07    3888
08    6343
09    3005
10     968
11    2263
12    1615
13    1598
14     641
15    1203
16    1620
17    1632
18     526
19    3089
20    3209
21    1031
22    2458
23    3381
24    2534
25    1021
26    1410
27    1768
28    1268
29     676
30    1285
31    1633
32    2646
33     567
34    2912
35    2798
36    1097
37    2768
38    1239

In [25]:
d_lvl2_bcat.head(5)

,id,date,latitude,longitude,description,sources,associations,a00,a01,a02,a03,a04,a05,a06,a07,a08,a09,a10,a11,undefined,w00,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w12,w13,w14,unknown,none,postcode,admin_unit,oblast,raion_nominatim_ua,raion_nominatim_en,raion_postcode
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,CIV0003,2022-02-24,50.470772,30.528098,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/TreyYingst/status/14967937...,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,04070,04,Kyiv,None,NaN,Kyiv
1,CIV0098,2022-02-24,49.212119,38.905921,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste...",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,92760,92,Luhansk Oblast,Старобільський район,Starobilsk Raion,Starobilskyi
2,CIV0013,2022-02-24,48.055395,37.778300,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System...",0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,83054,83,Donetsk Oblast,Донецький район,Donetsk Raion,<Rus-occupied>
3,CIV0004,2022-02-24,47.775609,37.239673,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/N_Waters89/status/14968566...,"Type of area affected=Healthcare,Weapon System...",0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,85670,85,Donetsk Oblast,Волноваський район,Volnovakha Raion,Volnovaskyi
4,CIV0011,2022-02-24,50.308310,34.880702,Civilian buildings damaged and destroyed by sh...,https://t.me/nexta_live/16696,"Type of area affected=Commercial,Weapon System...",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,42700,42,Sumy Oblast,Охтирський район,Okhtyrka Raion,<Rus-occupied>


In [26]:
# Validation: Compare raion mappings from both methods

# Check where both methods returned a result
both_valid = d_lvl2_bcat['raion_nominatim_ua'].notna() & (d_lvl2_bcat['raion_postcode'] != '<Rus-occupied>')
n_both = both_valid.sum()

print(f"Rows with both raion values: {n_both}/{len(d_lvl2_bcat)}")

if n_both > 0:
    # Compare results (case-insensitive, strip whitespace)
    d_lvl2_bcat['raion_match'] = d_lvl2_bcat.apply(
        lambda row: (
            str(row['raion_nominatim_ua']).lower().strip() == 
            str(row['raion_postcode']).lower().strip()
        ) if pd.notna(row['raion_nominatim_ua']) and row['raion_postcode'] != '<Rus-occupied>' else np.nan,
        axis=1
    )
    
    n_match = d_lvl2_bcat['raion_match'].sum()
    match_rate = n_match / n_both * 100 if n_both > 0 else 0
    
    print(f"Exact matches: {int(n_match)}/{n_both} ({match_rate:.1f}%)")
    
    # Show mismatches for inspection
    mismatches = d_lvl2_bcat[both_valid & (d_lvl2_bcat['raion_match'] == False)][
        ['postcode', 'latitude', 'longitude', 'raion_nominatim_ua', 'raion_postcode']
    ]
    if len(mismatches) > 0:
        print(f"\nMismatches ({len(mismatches)}):")
        mismatches.head(10)
    else:
        print("\nNo mismatches found!")

Rows with both raion values: 65/100
Exact matches: 0/65 (0.0%)

Mismatches (65):


,postcode,latitude,longitude,raion_nominatim_ua,raion_postcode
index,,,,,
1,92760,49.212119,38.905921,Старобільський район,Starobilskyi
3,85670,47.775609,37.239673,Волноваський район,Volnovaskyi
5,87503,47.117099,37.684831,Маріупольський район,Mariupolskyi
6,87503,47.120906,37.680990,Маріупольський район,Mariupolskyi
7,20305,48.748938,30.218889,Уманський район,Umanskyi
11,63501,49.850781,36.659762,Чугуївський район,Chuhuivskyi
14,08290,50.553864,30.275885,Бучанський район,Buchanskyi
16,61168,50.022621,36.330786,Харківський район,Kharkivskyi
17,08200,50.514217,30.227246,Бучанський район,Buchanskyi


In [27]:
#############################################################################################
# DEBUG: Inspect raw Nominatim response for a sample coordinate
import requests

# Use first row with valid coordinates
sample_row = d_lvl2_bcat[d_lvl2_bcat['latitude'].notna()].iloc[0]
lat, lon = sample_row['latitude'], sample_row['longitude']

print(f"Testing coordinates: ({lat}, {lon})")

params = {
    'lat': lat,
    'lon': lon,
    'format': 'jsonv2',
    'addressdetails': 1,
    }
headers = {'User-Agent': NOMINATIM_USER_AGENT}

resp = requests.get(
    'https://nominatim.openstreetmap.org/reverse',
    params=params,
    headers=headers,
    timeout=10,
    )
data = resp.json()

print(f"\nFull address breakdown:")
for key, value in data.get('address', {}).items():
    print(f"  {key}: {value}")

print(f"\nDisplay name: {data.get('display_name', 'N/A')}")
#############################################################################################

Testing coordinates: (50.470772, 30.528098)

Full address breakdown:
  road: Гаванський міст
  quarter: Рибальський острів
  suburb: Поділ
  borough: Подільський район
  city: Київ
  ISO3166-2-lvl4: UA-30
  postcode: 04070
  country: Україна
  country_code: ua

Display name: Гаванський міст, Рибальський острів, Поділ, Подільський район, Київ, 04070, Україна


In [28]:
# Export event-level data prior to aggregation
d_lvl2_bcat.to_csv(f'{DATA_PROC}/d_lvl2_bcat_event.csv', index=False)

In [29]:
#############################################################################################
#############################################################################################
# Restrict to n = 100 for geocoding tests
d_lvl2_ucdp = d_lvl2_ucdp.iloc[:100]
d_lvl2_ucdp.info()
#############################################################################################
#############################################################################################

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   conflict_name      100 non-null    object        
 1   source_article     100 non-null    object        
 2   source_office      100 non-null    object        
 3   source_headline    100 non-null    object        
 4   source_original    100 non-null    object        
 5   where_prec         100 non-null    int64         
 6   where_description  91 non-null     object        
 7   adm_1              90 non-null     object        
 8   adm_2              67 non-null     object        
 9   latitude           100 non-null    float64       
 10  longitude          100 non-null    float64       
 11  geom_wkt           100 non-null    object        
 12  priogrid_gid       100 non-null    int64         
 13  country            100 non-null    object        
 14  event_clari

In [30]:
#############################################################################################

### TODO 7/29: UCDP HERE - refactor this jangly mess

# Convert latitude/longitude coordinates → Ukrainian postcodes via Nominatim API
# Reuses get_postcode() function defined in Bellingcat section

postcodes_ucdp = []

for idx, row in tqdm(d_lvl2_ucdp.iterrows(), total=len(d_lvl2_ucdp), desc="Geocoding UCDP postcodes"):
    lat = row['latitude']
    lon = row['longitude']
    
    postcode = get_postcode(lat, lon)
    postcodes_ucdp.append(postcode)
    
    sleep(NOMINATIM_DELAY_SEC)

d_lvl2_ucdp['postcode'] = postcodes_ucdp

print(f"\nGeocoding complete!")
print(f"Postcodes found: {d_lvl2_ucdp['postcode'].notna().sum()}/{len(d_lvl2_ucdp)}")
#############################################################################################

Geocoding UCDP postcodes:   0%|          | 0/100 [00:00<?, ?it/s]


Geocoding complete!
Postcodes found: 78/100


In [31]:
#############################################################################################

### TODO 7/29: RELOCATE DOWNSTREAM

# Extract first 2 digits of postcode as `admin_unit` id
d_lvl2_ucdp['admin_unit'] = d_lvl2_ucdp['postcode'].astype(str).str[:2]

# Replace 'na' (from NaN conversion) with actual NaN
d_lvl2_ucdp.loc[d_lvl2_ucdp['admin_unit'] == 'na', 'admin_unit'] = np.nan

# Map `admin_unit` → `oblast`
d_lvl2_ucdp['oblast'] = d_lvl2_ucdp['admin_unit'].map(ADMIN_UNIT_TO_OBLAST)

# Verify mapping
print(f"Mapped oblasts: {d_lvl2_ucdp['oblast'].notna().sum()}/{len(d_lvl2_ucdp)}")
print(f"\nOblast distribution:")
d_lvl2_ucdp['oblast'].value_counts()
#############################################################################################

Mapped oblasts: 78/100

Oblast distribution:


oblast
Donetsk Oblast            14
Kharkiv Oblast            12
Kherson Oblast             9
Chernihiv Oblast           8
Mykolaiv Oblast            6
Kyiv Oblast                6
Zhytomyr Oblast            5
Luhansk Oblast             4
Sumy Oblast                4
Zaporizhzhia Oblast        4
Kyiv                       2
Cherkasy Oblast            2
Ivano-Frankivsk Oblast     1
Odesa Oblast               1
Name: count, dtype: int64

In [32]:
#############################################################################################

### TODO 7/29: RELOCATE DOWNSTREAM

# Map latitude/longitude coordinates → Ukrainian raions via Nominatim API
# Reuses raion_from_point_nominatim() function defined in Bellingcat section

raions_ucdp = []

for idx, row in tqdm(d_lvl2_ucdp.iterrows(), total=len(d_lvl2_ucdp), desc="Geocoding UCDP raions"):
    raion = raion_from_point_nominatim(
        row['latitude'],
        row['longitude'],
        user_agent=NOMINATIM_USER_AGENT,
    )
    raions_ucdp.append(raion)

d_lvl2_ucdp['raion_nominatim_ua'] = raions_ucdp

print(f"\nRaion geocoding complete!")
print(f"Raions found: {d_lvl2_ucdp['raion_nominatim_ua'].notna().sum()}/{len(d_lvl2_ucdp)}")
#############################################################################################

Geocoding UCDP raions:   0%|          | 0/100 [00:00<?, ?it/s]


Raion geocoding complete!
Raions found: 98/100


In [33]:
#############################################################################################

### TODO 7/29: RELOCATE DOWNSTREAM

# Map Ukrainian raion names → English translations
d_lvl2_ucdp['raion_nominatim_en'] = d_lvl2_ucdp['raion_nominatim_ua'].map(RAION_UA_TO_EN)

# Report coverage
mapped = d_lvl2_ucdp['raion_nominatim_en'].notna().sum()
total_with_ua = d_lvl2_ucdp['raion_nominatim_ua'].notna().sum()
print(f"Mapped to English: {mapped}/{total_with_ua}")

# Show any unmapped Ukrainian raions for dictionary updates
unmapped = d_lvl2_ucdp[d_lvl2_ucdp['raion_nominatim_ua'].notna() & d_lvl2_ucdp['raion_nominatim_en'].isna()]['raion_nominatim_ua'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped raions (add to mappings.py):")
    for r in unmapped:
        print(f"    '{r}': '',")

print(f"\nSample results:")
d_lvl2_ucdp[['latitude', 'longitude', 'oblast', 'raion_nominatim_ua', 'raion_nominatim_en']].head(10)
#############################################################################################

Mapped to English: 96/98

Unmapped raions (add to mappings.py):
    'Сіверськодонецький район': '',

Sample results:


,latitude,longitude,oblast,raion_nominatim_ua,raion_nominatim_en
0,50.272064,35.980068,Kharkiv Oblast,Богодухівський район,Bohodukhiv Raion
1,47.145783,37.584772,Donetsk Oblast,Маріупольський район,Mariupol Raion
2,46.959201,31.987586,Mykolaiv Oblast,Миколаївський район,Mykolaiv Raion
3,50.108463,36.115225,Kharkiv Oblast,Харківський район,Kharkiv Raion
4,49.980810,36.252720,Kharkiv Oblast,Харківський район,Kharkiv Raion
5,51.000000,30.500000,NaN,Вишгородський район,Vyshhorod Raion
6,50.272064,35.980068,Kharkiv Oblast,Богодухівський район,Bohodukhiv Raion
7,46.614720,31.545050,Mykolaiv Oblast,Миколаївський район,Mykolaiv Raion
8,48.736823,39.228735,Luhansk Oblast,Щастинський район,Shchastia Raion
9,46.640600,32.615890,Kherson Oblast,Херсонський район,Kherson Raion


In [34]:
d_lvl2_ucdp.to_csv('d_lvl2_ucdp_geo_inspect.csv')

### b. Collapse / export: Bellingcat OSINT Civilian Harm in Ukraine

In [35]:
# Aggregate at raion level

### Collapses event-level data to raion-level counts
### Uses raion_nominatim_en as primary raion identifier

# Define all dummy variables to aggregate
area_type_vars = ['a00', 'a01', 'a02', 'a03', 'a04', 'a05', 
                  'a06', 'a07', 'a08', 'a09', 'a10', 'a11', 
                  'undefined']

weapon_sys_vars = ['w00', 'w01', 'w02', 'w03', 'w04', 'w05',
                   'w06', 'w07', 'w08', 'w09', 'w10', 'w11', 
                   'w12', 'w13', 'w14', 'unknown', 'none']

all_dummy_vars = area_type_vars + weapon_sys_vars

# Build aggregation dict: sum all dummy vars, count events
agg_dict = {var: 'sum' for var in all_dummy_vars}
agg_dict['id'] = 'count'  # count events per raion

# Aggregate by raion (English name)
d_lvl2_bcat_raion = d_lvl2_bcat.groupby('raion_nominatim_en', as_index=False).agg(agg_dict)

# Rename id count column
d_lvl2_bcat_raion = d_lvl2_bcat_raion.rename(columns={'id': 'n_events'})

# Add oblast mapping (most common oblast per raion)
oblast_map = d_lvl2_bcat.groupby('raion_nominatim_en')['oblast'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
)
d_lvl2_bcat_raion['oblast'] = d_lvl2_bcat_raion['raion_nominatim_en'].map(oblast_map)

# Reorder columns: raion, oblast, n_events, area types, weapon systems
col_order = ['raion_nominatim_en', 'oblast', 'n_events'] + area_type_vars + weapon_sys_vars
d_lvl2_bcat_raion = d_lvl2_bcat_raion[col_order]

# Sort by total events (descending)
d_lvl2_bcat_raion = d_lvl2_bcat_raion.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Raion-level aggregation: {len(d_lvl2_bcat_raion)} raions")
print(f"Total events: {d_lvl2_bcat_raion['n_events'].sum():,}")
print(f"\nTop 10 raions by event count:")
d_lvl2_bcat_raion.head(10)

KeyError: "Column(s) ['w11'] do not exist"

In [ ]:
# Export raion-level data for 1:n merge
d_lvl2_bcat_raion.to_csv(f'{DATA_PROC}/d_lvl2_bcat_raion.csv', index=False)

In [ ]:
# Aggregate at oblast level

### Collapses event-level data to oblast-level counts

# Build aggregation dict: sum all dummy vars, count events
agg_dict_obl = {var: 'sum' for var in all_dummy_vars}
agg_dict_obl['id'] = 'count'

# Aggregate by oblast
d_lvl2_bcat_oblast = d_lvl2_bcat.groupby('oblast', as_index=False).agg(agg_dict_obl)

# Rename id count column
d_lvl2_bcat_oblast = d_lvl2_bcat_oblast.rename(columns={'id': 'n_events'})

# Add raion count per oblast
raion_counts = d_lvl2_bcat.groupby('oblast')['raion_nominatim_en'].nunique()
d_lvl2_bcat_oblast['n_raions'] = d_lvl2_bcat_oblast['oblast'].map(raion_counts)

# Reorder columns: oblast, n_raions, n_events, area types, weapon systems
col_order_obl = ['oblast', 'n_raions', 'n_events'] + area_type_vars + weapon_sys_vars
d_lvl2_bcat_oblast = d_lvl2_bcat_oblast[col_order_obl]

# Sort by total events (descending)
d_lvl2_bcat_oblast = d_lvl2_bcat_oblast.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Oblast-level aggregation: {len(d_lvl2_bcat_oblast)} oblasts")
print(f"Total events: {d_lvl2_bcat_oblast['n_events'].sum():,}")
print(f"\nOblast counts:")
d_lvl2_bcat_oblast

In [ ]:
# Export oblast-level data for (potential) 1:n merge
d_lvl2_bcat_oblast.to_csv(f'{DATA_PROC}/d_lvl2_bcat_oblast.csv', index=False)